## Create sequential Models based on neural network Using pytorch

In [ ]:
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
# from torchsummary import summary 
from torchinfo import summary
from torch.utils.data import DataLoader
import torch.optim as optim
import math

In [ ]:
# Flag to control normalization
apply_normalize = True   # set to False if you don’t want normalization

# 1. Define preprocessing transforms
if apply_normalize:
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))  # MNIST mean & std
    ])
else:
    transform = transforms.Compose([
        transforms.ToTensor()  # just scale [0,255] → [0,1]
    ])

# 2. Load MNIST dataset
train_dataset = torchvision.datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.MNIST(
    root="./data", train=False, download=True, transform=transform
)

# 3. Create DataLoaders
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 4. Inspect shapes
images, labels = next(iter(train_loader))
print(f"X_train batch shape: {images.shape}")   # (batch_size, 1, 28, 28)
print(f"y_train batch shape: {labels.shape}")   # (batch_size,)

In [ ]:
# =========================
# Global configuration vars
# =========================
USE_BIAS = True
KERNEL_INIT = "glorot_uniform"   # maps to nn.init.xavier_uniform_
BIAS_INIT = "zeros"              # maps to nn.init.zeros_

# Regularization placeholders
KERNEL_REGULARIZER = None
BIAS_REGULARIZER = None
ACTIVITY_REGULARIZER = None
KERNEL_CONSTRAINT = None
BIAS_CONSTRAINT = None

# LoRA parameters (not natively in PyTorch Dense)
LORA_RANK = None
LORA_ALPHA = None

# Quantization config placeholder
QUANTIZATION_CONFIG = None

# =========================
# Model definition
# =========================
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        
        # Flatten layer
        self.flatten = nn.Flatten()
        
        # Dense equivalents
        self.fc1 = nn.Linear(28*28, 300, bias=USE_BIAS)
        self.fc2 = nn.Linear(300, 150, bias=USE_BIAS)
        self.fc3 = nn.Linear(150, 10, bias=USE_BIAS)
        
        # Apply initializers
        self._init_weights()
    
    def _init_weights(self):
        if KERNEL_INIT == "glorot_uniform":
            nn.init.xavier_uniform_(self.fc1.weight)
            nn.init.xavier_uniform_(self.fc2.weight)
            nn.init.xavier_uniform_(self.fc3.weight)
        if BIAS_INIT == "zeros":
            nn.init.zeros_(self.fc1.bias)
            nn.init.zeros_(self.fc2.bias)
            nn.init.zeros_(self.fc3.bias)
    
    def forward(self, x):
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.softmax(self.fc3(x), dim=1)
        return x

In [ ]:
# =========================
# Instantiate model
# =========================
# --- Model (defined ) ---
torch.manual_seed(42)
model_no_normalizations_nn = SimpleNN()
print(model_no_normalizations_nn)

In [ ]:
# Equivalent to Keras model.summary()
summary(model_no_normalizations_nn, input_size=(1, 28, 28))

In [ ]:
# --- Optimizer (keras: optimizer="Adam") ---
optimizer = optim.Adam(model_no_normalizations_nn.parameters(), lr=0.001)

# --- Loss (keras: loss="CategoricalCrossentropy") ---
loss_fn = nn.CrossEntropyLoss()

# --- Accuracy metric ---
def accuracy(y_pred, y_true):
    return (y_pred.argmax(dim=1) == y_true).float().mean()

# --- AMP (auto_scale_loss=True) ---
scaler = torch.cuda.amp.GradScaler()

In [ ]:
# --- Training parameters ---
batch_size_  = 64
epochs_      = 30
sample_size_ = len(train_dataset)
Validation_sample_set = len(test_dataset)
validation_batch_size_ = batch_size_
validation_freq_ = 1

#--- Training loop (equivalent to model.fit) ---
for epoch in range(epochs_):
    model_no_normalizations_nn.train()
    for x, y in train_loader:
        optimizer.zero_grad()
        
        # Mixed precision training
        with torch.cuda.amp.autocast():
            y_pred = model_no_normalizations_nn(x)
            loss = loss_fn(y_pred, y)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
    
    # Validation (every epoch, like validation_freq=1)
    if (epoch + 1) % validation_freq_ == 0:
        model_no_normalizations_nn.eval()
        val_loss, val_acc = 0.0, 0.0
        with torch.no_grad():
            for x_val, y_val in test_loader:
                y_pred_val = model_no_normalizations_nn(x_val)
                val_loss += loss_fn(y_pred_val, y_val).item()
                val_acc += accuracy(y_pred_val, y_val).item()
        val_loss /= len(test_loader)
        val_acc /= len(test_loader)
        print(f"Epoch {epoch+1}/{epochs_} - Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")